# 10 · Data Modeling — Silver → **Gold** (star schema)

**Gold** is the business-ready layer: data shaped for consumption — **dimensions**
and **facts** in a **star schema**, plus aggregated **marts** for dashboards.
We turn `brewbox.orders_silver` (from notebook 9) and the reference tables into a
clean model BI and ML can query directly.

Recap from the intro: **facts** = the measurable events (one row per event at a
grain, with foreign keys + numeric measures); **dimensions** = the descriptive
context (who/what/where/when).

In [ ]:
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
from pyspark.sql import functions as F
spark.sql("USE SCHEMA brewbox")
print("Silver orders:", spark.table("brewbox.orders_silver").count())

## 1 · Build the dimensions

Each dimension is one row per entity with descriptive attributes. We build
`dim_customer`, `dim_product`, `dim_store`, and a `dim_date` derived from the
order dates.

In [ ]:
dim_customer = spark.table("brewbox.customers").select(
    "customer_id","name","country","loyalty_tier","signup_date")
dim_product = spark.table("brewbox.products").select(
    "product_id","product_name","category","unit_price")
dim_store = spark.table("brewbox.stores").select(
    "store_id","store_name","city","region")

dim_date = (spark.table("brewbox.orders_silver")
    .select(F.col("order_date").alias("date")).distinct()
    .withColumn("year",  F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day",   F.dayofmonth("date"))
    .withColumn("weekday", F.date_format("date","EEEE")))

for name, dfx in [("dim_customer",dim_customer),("dim_product",dim_product),
                  ("dim_store",dim_store),("dim_date",dim_date)]:
    dfx.write.format("delta").mode("overwrite").saveAsTable(f"brewbox.{name}")
    print(f"brewbox.{name}:", spark.table(f"brewbox.{name}").count(), "rows")

## 2 · Build the fact tables

**`fct_orders`** — grain: one row per order, with foreign keys to the dimensions
and the `amount` measure. **`fct_order_items`** — grain: one row per order line,
with `quantity` and a computed `line_amount`, joined to products for category.

In [ ]:
# fct_orders (order grain) - straight from Silver, keeping keys + measures
fct_orders = spark.table("brewbox.orders_silver").select(
    "order_id","customer_id","store_id","order_date","status","is_completed","amount")
fct_orders.write.format("delta").mode("overwrite").saveAsTable("brewbox.fct_orders")

# fct_order_items (line grain) - items + product category + line_amount + order date/region
items = spark.table("brewbox.order_items")
fct_order_items = (items
    .join(spark.table("brewbox.dim_product").select("product_id","category"), "product_id", "left")
    .join(spark.table("brewbox.orders_silver").select("order_id","order_date","region","is_completed"),
          "order_id", "left")
    .withColumn("line_amount", F.round(F.col("quantity") * F.col("unit_price"), 2)))
fct_order_items.write.format("delta").mode("overwrite").saveAsTable("brewbox.fct_order_items")

print("fct_orders:", spark.table("brewbox.fct_orders").count(),
      "| fct_order_items:", spark.table("brewbox.fct_order_items").count())

## 3 · Query the star — join fact to dimensions

This is the payoff of the model: answer business questions by joining the fact to
the dimensions it references. "Completed revenue by region and month":

In [ ]:
(spark.table("brewbox.fct_orders").filter("is_completed")
    .join(spark.table("brewbox.dim_store"), "store_id")
    .withColumn("month", F.date_format("order_date","yyyy-MM"))
    .groupBy("region","month")
    .agg(F.round(F.sum("amount"),2).alias("revenue"))
    .orderBy("region","month")
    .show(10))

## 4 · Build the Gold marts (aggregated, business-ready)

Marts pre-aggregate the star for specific dashboards, so BI queries are instant.

In [ ]:
# Daily revenue by region (order grain)
mart_daily_revenue = (spark.table("brewbox.fct_orders").filter("is_completed")
    .join(spark.table("brewbox.dim_store").select("store_id","region"), "store_id")
    .groupBy("order_date","region")
    .agg(F.count("*").alias("orders"), F.round(F.sum("amount"),2).alias("revenue")))
mart_daily_revenue.write.format("delta").mode("overwrite").saveAsTable("brewbox.mart_daily_revenue")

# Revenue by product category (line grain)
mart_category_revenue = (spark.table("brewbox.fct_order_items").filter("is_completed")
    .groupBy("category")
    .agg(F.round(F.sum("line_amount"),2).alias("revenue"))
    .orderBy(F.desc("revenue")))
mart_category_revenue.write.format("delta").mode("overwrite").saveAsTable("brewbox.mart_category_revenue")

spark.table("brewbox.mart_category_revenue").show()

## 5 · Star vs snowflake (recap)

Our Gold is a **star schema**: `fct_orders` in the middle, flat dimensions around
it. A **snowflake** would normalize a dimension further — e.g. split
`dim_product`'s `category` into its own `dim_category` table. Star = fewer joins,
simpler, faster (the usual BI choice); snowflake = less redundancy for large,
complex dimensions.

```
        dim_date
            |
 dim_store --- fct_orders --- dim_customer
                   |
              (measures: amount)
```

## 6 · Exercises

**Exercise 1 —** Top 5 customers by completed revenue (join `fct_orders` to
`dim_customer`).

In [ ]:
# Your turn (Exercise 1):

In [ ]:
# ✅ Solution 1
(spark.table("brewbox.fct_orders").filter("is_completed")
    .join(spark.table("brewbox.dim_customer"), "customer_id")
    .groupBy("customer_id","name")
    .agg(F.round(F.sum("amount"),2).alias("revenue"))
    .orderBy(F.desc("revenue")).show(5))

**Exercise 2 —** Using `dim_date`, how many orders happened on each weekday?

In [ ]:
# Your turn (Exercise 2):

In [ ]:
# ✅ Solution 2
(spark.table("brewbox.fct_orders")
    .join(spark.table("brewbox.dim_date"), spark.table("brewbox.fct_orders").order_date == spark.table("brewbox.dim_date").date)
    .groupBy("weekday").count().orderBy(F.desc("count")).show())

**Exercise 3 —** From `brewbox.mart_category_revenue`, what share of total revenue
does each category represent? (Add a `pct` column.)

In [ ]:
# Your turn (Exercise 3):

In [ ]:
# ✅ Solution 3
from pyspark.sql import Window
tot = Window.partitionBy()
(spark.table("brewbox.mart_category_revenue")
    .withColumn("pct", F.round(F.col("revenue")/F.sum("revenue").over(tot)*100, 1))
    .show())

## 7 · Recap & next

You built a **Gold star schema** — `dim_customer`/`dim_product`/`dim_store`/
`dim_date` + `fct_orders`/`fct_order_items` — and aggregated **marts** ready for
BI. That completes a batch **Bronze → Silver → Gold** medallion pipeline on the
BrewBox data.

**Next → `11` Structured Streaming:** do the same medallion flow **continuously**,
as data arrives. 🚀